We consider the following stopping problem. We have a semi-infinite series of Gaussian buttons, which means that when we press each button we get some reward $r$ drawn from a Gaussian distribution with some mean $m$ and variance $\sigma_r$. 

$$
P(x \leq r < x + dx |m, \sigma_r) = G(x |m, \sigma_r)\, dx = \frac{1}{\sqrt{2\pi} \sigma_r} \exp\left[- \frac{(x - m)^2}{2\sigma_r^2}\right]\, dx.
$$

We assume that the variance $\sigma_r$ is a priori known to us and is the same for all buttons and the mean $m$ is unkhown and differs from button to button.
Starting with the first button at each given step we can either press the current button and observe the outcome or move on and press the next one. We are not allowed to return to previous buttons.

We take the Bayessian approach to estimating the probability of a Gaussian button, namely, our knowledge about Gaussian buttons means is given by a Gaussian distribution

$$
P(m |\mu, \sigma) = G(m |\mu, \sigma) = \frac{1}{\sqrt{2\pi} \sigma} \exp\left[- \frac{(m - \mu)^2}{2\sigma^2}\right].
$$

We suppose that when we move to a new button we start with a prior $\mu = \mu_0$, $\sigma = \sigma_0$. 

Now suppose we observe an outcome $x$ drawn from the Gaussian distribution with the variance $\sigma_r$ and the mean $m$

$$
P(x|m, \sigma_r) = G(x| m, \sigma_r),
$$

and we are given a Gaussian prior on $m$ 

$$
P(m|\mu, \sigma) = G(m| \mu, \sigma),
$$

From the basic Bayessian inference we know that in that case the posterior probability distribution for $m$ is a Gaussian distribution

$$
\begin{split}
&P(m |x ) = \frac{P(x |m ) P(m)}{P(x)} = G(m| \hat{\mu}, \hat{\sigma}),\\
&\frac{1}{\hat{\sigma}^2} = \frac{1}{\sigma^2} + \frac{1}{\sigma_r^2},\qquad \hat{\mu} = \hat{\sigma}^2\left(\frac{\mu}{\sigma^2} + \frac{x}{\sigma_r^2}\right).
\end{split}
$$

Moreover, the the probability distribution for $x$ given the prior is also Gaussian

$$
P(x) = \int P(x| m) P(m) dx = G\big(x| \mu, \sqrt{\sigma^2 + \sigma_r^2}\big).
$$

Combining it with our model for Gaussian button we get that after $n$ presses $(n\geq 0)$ with outcomes $\{x_k\}$ the posterior on the mean will be the following Gaussian

$$
P(m |\{x_k\}) = G\left(m |\mu(\{x_k\}) , \sigma_n \right),
$$

where

$$
\sigma_n = \frac{\sigma_r}{\sqrt{n + (\sigma_r/\sigma_0)^2}},\qquad \mu(\{x_k\}) = \frac{\mu_0}{1 + n (\sigma_0/\sigma_r)^2}+\frac{1}{n + (\sigma_r/\sigma_0)^2}\sum_{k = 1}^n x_k.
$$

We model the problem as a Markov Decision Process, thus we search for an optimal policy $\pi(s)$ that maximizes the expected reward

$$
V(s, T) = \mathrm{E}\left[\sum_{t = 1}^T r_{a^\pi} (s_t^{\pi}) \right].
$$

Here $s_t$ denotes our knowledge of the button we are pressing at time $t$ (its history of rewards)  and $T$ is the total number of button presses we are allowed. This knowledge can be summarized with a pair $(n, \mu)$, where $n$ is the number of presses we made for this button so far and $\mu = \mu(\{x_k\})$ is the observed posterior mean we defined earlier.
We solve this problem via dynamic programming. The formal solution can be written as

$$
V(s, T) = \max_{a\in A}\left(r_a (s) + \sum_{s'\in S} p_a (s, s') V(s', T-1)\right),
$$

where $a$ is a particular desicion to stay or to move on, $A$ is the space of our actions, $r_a(s)$ is the expected reward for taking an action and $V(s, 0) = 0$. Our space of states $S$ consists of our knowledge about the current button we are pressing, so it can be described with a pair $(n, \mu)$ corresponding to a Gaussian distribution. Our space of actions consists of either pressing the current button or moving to the next button and pressing it. If we decide to stay with the current button we can expect to get a reward $\mu$ and with the probability distributed as $G(x|\mu, \sqrt{\sigma_n^2 + \sigma_r^2})$ move to the state with an updated posterior mean $\hat{\mu} = \sigma^2_{n+1}(\mu/\sigma_n^2 + x/\sigma_r^2)$. If we decide to move to the next button we get no expected reward (due to our prior) and move with the probability distributed as $G(x|\mu_0, \sqrt{\sigma_0^2 + \sigma_r^2})$ to the state with posterior mean $\hat{\mu} = \mu_0/(1 + (\sigma_0/\sigma_r)^2) + x/(1 + (\sigma_r/\sigma_0)^2)$

Labeling the expected reward from a button we pressed $n$ times before with the posterior mean $\mu$ and the remaining number of steps $T$ as $V(\mu, n, T)$ we get the following recurrence relation

$$
V(\mu, n, T) = \max \left\{
\begin{matrix}
\mu + \int G(x|\mu, \sqrt{\sigma_n^2 + \sigma_r^2})\, V\left(\sigma_{n+1}^2\left(\frac{\mu}{\sigma_n^2} + \frac{x}{\sigma_r^2}\right), n+1, T-1\right)dx\\
\int G(x|\mu_0, \sqrt{\sigma_0^2 + \sigma_r^2})\, V\left(\frac{\mu_0}{1 + (\sigma_0/\sigma_r)^2} + \frac{x}{1 + (\sigma_r/\sigma_0)^2}, 1, T-1\right)dx
\end{matrix}\right.,
$$

and $V(\mu, n, 0) = 0$. We also note that according to this definition 

$$
V(0, 0, T) = \int G(x|\mu_0, \sqrt{\sigma_0^2 + \sigma_r^2})\, V\left(\frac{\mu_0}{1 + (\sigma_0/\sigma_r)^2} + \frac{x}{1 + (\sigma_r/\sigma_0)^2}, 1, T-1\right)dx
$$

The optimal policy thus corresponds to comparing

$$
\mu + \int G(x|\mu, \sqrt{\sigma_n^2 + \sigma_r^2})\, V\left(\sigma_{n+1}^2\left(\frac{\mu}{\sigma_n^2} + \frac{x}{\sigma_r^2}\right), n+1, T-1\right)dx,
$$ 

to $V(0, 0, T)$ and moving to the next button if the latter is greater.